# Privan-130M: Pretraining and Loss Optimization

This notebook covers:
1. Decoupled AdamW optimizer configuration
2. Learning rate warmup and cosine annealing scheduler
3. Next-token prediction loss computation
4. Overfitting validation step

In [ ]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path.cwd().parent / "src"))

from llm.config import Config
from llm.model import CausalLM
from llm.training.optimizer import create_optimizer
from llm.training.scheduler import CosineWarmupScheduler

## 1. Inspect Optimizer Weight Decay Separation

In [ ]:
cfg = Config.from_yaml("../configs/debug.yaml")
model = CausalLM(cfg.model)
optimizer = create_optimizer(model, cfg.optimizer)

print("Optimizer parameter groups:")
for i, group in enumerate(optimizer.param_groups):
    print(f"  Group {i}: weight_decay = {group['weight_decay']}, num_tensors = {len(group['params'])}")

## 2. Simulate Learning Rate Warmup & Cosine Decay

In [ ]:
scheduler = CosineWarmupScheduler(optimizer, warmup_steps=20, max_steps=100, min_lr=1e-5)
lrs = []
for step in range(100):
    optimizer.step()
    scheduler.step()
    lrs.append(scheduler.get_last_lr()[0])

print(f"Step 0 LR:   {lrs[0]:.2e}")
print(f"Step 20 LR:  {lrs[19]:.2e} (Peak)")
print(f"Step 100 LR: {lrs[-1]:.2e} (Min LR)")